In [20]:
import os
import sys
import polars as pl
from polars import Expr
from pathlib import Path

import warnings

warnings.simplefilter("ignore")

#### Config

In [21]:
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "data" / "games.csv").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot find data/games.csv")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from steam_data import OWNER_COLUMNS, add_estimated_owner_columns, ownership_target
DATA_PATH = PROJECT_ROOT / "data"
GAMES_PATH = DATA_PATH / "games.csv"

### data loading and cleaning

In [22]:
# data loading and cleaning part 1: read csv


df = pl.read_csv(GAMES_PATH)

In [23]:
# data loading and cleaning part 2: rename columns
df = df.rename(
    {
        "AppID": "steam_appid",
        "Developers": "developers",
        "Publishers": "publishers",
        "Categories": "categories",
        "Genres": "genres",
        "Tags": "tags",
    }
)

In [24]:
# data loading and cleaning part 3: change release date datatype to pl.Date


def expr_release_date_to_date(e: pl.Expr):
    """Turns the release date from a string representation to a `pl.Date` representation.

    Some preprocessing of the column is needed before making this conversion, because
    the release date can be of the following two forms:
    1. Apr 27, 2014
    2. Apr 2014

    In the second case we first transform the string representation to
    Apr 1, 2014 before converting the whole column to a `pl.Date`
    """

    month_regex = (
        r"\b(?<month>Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec) (?<year>\d{4}\b)"
    )

    release_date_name = e.meta.output_name()
    release_date_groups = e.str.extract_groups(month_regex).alias(
        f"{release_date_name} groups"
    )
    release_date_str = (
        pl.when(e.str.contains(month_regex))
        .then(
            pl.concat_str(
                release_date_groups.struct["month"],
                pl.lit(" 1, "),
                release_date_groups.struct["year"],
            )
        )
        .otherwise(e)
    )
    release_date = release_date_str.str.to_date("%b %d, %Y").alias(release_date_name)

    return release_date


df = df.with_columns(expr_release_date_to_date(pl.col("Release date")))

In [25]:
# data loading and cleaning part 4: turn columns with lists from string into List


def _expr_str_split_list_column(c: Expr):
    name = c.meta.output_name()
    return pl.when(c == "").then(pl.lit([])).otherwise(c.str.split(",")).alias(name)


cols_to_str_split = ["developers", "publishers", "categories", "genres", "tags"]
cols = [_expr_str_split_list_column(pl.col(c)) for c in cols_to_str_split]

df = df.with_columns(*cols)

In [26]:
# data loading and cleaning part 5: Sort DataFrame by 'steam_appid'
df = df.sort("steam_appid")

### Feature engineering

In [27]:
# Shared definitions: lb = lower endpoint; ub = upper endpoint; mid = (lb + ub) / 2.
# Define all three BEFORE filtering, target construction or inspection.
DATA_OBSERVATION_DATE = pl.date(2023, 7, 5)
owner_columns = add_estimated_owner_columns(df.select("Estimated owners").to_pandas())[OWNER_COLUMNS]
df = df.with_columns(pl.from_pandas(owner_columns).get_columns())

In [28]:
# feature engineering part 2: Add 'year' feature based on 'Release date'
df = df.with_columns(
    (DATA_OBSERVATION_DATE - df["Release date"])
    .map_elements(lambda x: x.days / 365)
    .alias("year")
)

In [29]:
# feature engineering part 3: Calculate 'Steam score' based on positive and negative reviews
df = df.with_columns(
    (df["Positive"] / (df["Positive"] + df["Negative"])).alias("Steam score")
)

In [30]:
# feature engineering part 4: One-hot encode 'tags' and 'genres'

from sklearn.preprocessing import MultiLabelBinarizer


def add_multilabel_onehot_encoded_column(df: pl.DataFrame, col):
    """Add a one-hot-encoded column to the dataframe from a multi-label column."""
    mlb = MultiLabelBinarizer()
    encoded_data = mlb.fit_transform(df[col])
    feature_names = [f"{col[:-1]}_{c}" for c in list(mlb.classes_)]
    encoded_df = pl.DataFrame(encoded_data, schema=list(feature_names))
    return df.with_columns(encoded_df)


df = add_multilabel_onehot_encoded_column(df, "tags")
df = add_multilabel_onehot_encoded_column(df, "genres")

In [31]:
# feature engineering part 5: Sort dataframe by 'steam_appid'
df = df.sort("steam_appid")

### Filtering

In [32]:
# filtering part 1: filter out games less than a year old


def filter_out_new_games(
    df: pl.DataFrame,
    min_age: str = "1y",
    observation_date: pl.Expr = DATA_OBSERVATION_DATE,
) -> pl.DataFrame:
    """
    We want games that are at least a year old so that the downloads/purchases
    have had the chance to stabilize.
    """
    max_release_date = observation_date.dt.offset_by(f"-{min_age}")
    df = df.filter(pl.col("Release date") <= max_release_date)

    return df


df = filter_out_new_games(df)

In [33]:
# filtering part 2: filter out games with 0 estimated owners

ESTIMATED_OWNERS_POSITIVE_FILTER = pl.col("Estimated owners (mid)") > 0.0
"""Filter applied in `clean_data`. These are very unpopular games, with worse data
quality because not many players have played it and voted for their tags.
Moreover, we want the estimated owners (mid) to be positive
so that we can take the log. Typically, these are also games that have fewer user
reviews, and hence a less reliable user score (this can e.g. be a target for
modelling)"""

df = df.filter(ESTIMATED_OWNERS_POSITIVE_FILTER)

In [34]:
# filtering part 3: sort by steam_appid
df = df.sort("steam_appid")

### Adjusting the target feature of the dataset

In [35]:
# Deterministic ownership bands; never assign target IDs from unsorted unique values.
target = ownership_target(df.select(OWNER_COLUMNS).to_pandas())
df = df.with_columns(pl.Series("target", target.to_numpy()))

In [36]:
# The shared target is already bucketed: 0 (<20k), 1 (20k–<100k), 2 (100k+).
assert df["target"].is_in([0, 1, 2]).all()
assert (df["Estimated owners (lb)"] <= df["Estimated owners (mid)"]).all()
assert (df["Estimated owners (mid)"] <= df["Estimated owners (ub)"]).all()

### Exporting the DF

In [37]:
def write_parquet(df: pl.DataFrame, data_path: str | Path = DATA_PATH):
    games_path = data_path / "games.parquet"
    df.write_parquet(games_path)

    print(f"Dataset saved at {games_path}")


write_parquet(df)

Dataset saved at /mnt/c/Workspace3/machine-learning-engineer-amsterdam (29)/data/games.parquet


#### Inspect bins

In [38]:
def inspect_target_and_owners(df: pl.DataFrame) -> pl.DataFrame:
    """
    Inspect the relation between 'target' column and the estimated owners columns.
    """
    return (
        df[["target", "Estimated owners (lb)", "Estimated owners (ub)"]]
        .unique()
        .sort(by=["target", "Estimated owners (lb)"])
    )


# based on this, the target bins are '0 - 20k estimated owners', '20k - 100k estimated owners', and '100k to 200 million estimated owners'
inspect_target_and_owners(df)

target,Estimated owners (lb),Estimated owners (ub)
i64,f64,f64
0,0.0,20000.0
1,20000.0,50000.0
1,50000.0,100000.0
2,100000.0,200000.0
2,200000.0,500000.0
…,…,…
2,5e6,1e7
2,1e7,2e7
2,2e7,5e7
